# Python vs NumPy: Vectorization, Chunking, and Parallelism

We now compare three ways to perform the same numerical computation:

1. **Pure Python**
2. **NumPy vectorization**
3. **Chunked NumPy**

The goal is to understand two important ideas:

- why NumPy can be much faster than a Python loop;
- why splitting a large computation into independent **chunks** is useful for memory management and makes the work easier to parallelize.

## 1. Setup

We use the same numerical task in every implementation:

```python
(i % 7) * (i % 13)
```

for all integers from `0` to `N-1`, and then sum the results.

In [9]:
import time
import numpy as np

N = 200_000_000

print(f"N = {N:,}")
print("NumPy version:", np.__version__)

N = 200,000,000
NumPy version: 2.5.2


## 2. Pure Python

This version executes the loop directly in Python.

Each iteration requires Python to:

1. retrieve `i`;
2. compute two modulo operations;
3. multiply the values;
4. update `total`;
5. repeat.

For millions of iterations, the interpreter overhead becomes important.

In [10]:
def vanilla_python(n):
    total = 0

    for i in range(n):
        total += (i % 7) * (i % 13)

    return total

In [11]:
start = time.perf_counter()

result_python = vanilla_python(N)

python_time = time.perf_counter() - start

print("Result:", result_python)
print(f"Pure Python: {python_time:.4f} s")

Result: 3599999947
Pure Python: 10.6773 s


## 3. NumPy vectorization

Instead of asking Python to process one element at a time, we create a NumPy array and describe the operation on the entire array.

The heavy work is delegated to optimized compiled numerical code.

In [12]:
def numpy_full(n):
    x = np.arange(n, dtype=np.int64)
    return np.sum((x % 7) * (x % 13))

In [13]:
start = time.perf_counter()

result_numpy = numpy_full(N)

numpy_time = time.perf_counter() - start

print("Result:", int(result_numpy))
print(f"NumPy: {numpy_time:.4f} s")
print(f"Speedup vs Python: {python_time / numpy_time:.2f}x")

Result: 3599999947
NumPy: 2.2975 s
Speedup vs Python: 4.65x


### Why is NumPy usually faster?

Pure Python:

```text
Python loop
   ↓
Python interpreter
   ↓
one element at a time
```

NumPy:

```text
Python
   ↓
one high-level array operation
   ↓
optimized compiled implementation
   ↓
many elements processed efficiently
```

The speedup is mostly due to reducing per-element Python overhead.

## 4. The memory cost of full vectorization

The full NumPy version creates an array with `N` elements.

Let's inspect its approximate memory footprint.

In [14]:
x = np.arange(N, dtype=np.int64)

print(f"Elements: {x.size:,}")
print(f"Bytes per element: {x.itemsize}")
print(f"Total memory: {x.nbytes / 1e6:.1f} MB")

del x

Elements: 200,000,000
Bytes per element: 8
Total memory: 1600.0 MB


For larger datasets, the array itself and intermediate expressions can consume substantial memory.

This motivates **chunking**.

## 5. Chunked NumPy

Instead of processing all `N` values at once, we split the interval into smaller independent blocks.

For example:

```text
0 ------------------------------------------------------------------ N

|------ chunk 1 ------|
                       |------ chunk 2 ------|
                                              |------ chunk 3 ------|
```

Each chunk can be processed with NumPy, and we only keep one chunk in memory at a time.

In [15]:
def numpy_chunked(n, chunk_size=1_000_000):
    total = 0

    for start_idx in range(0, n, chunk_size):
        stop_idx = min(start_idx + chunk_size, n)

        x = np.arange(start_idx, stop_idx, dtype=np.int64)

        partial_sum = np.sum((x % 7) * (x % 13))

        total += partial_sum

    return total

In [16]:
CHUNK_SIZE = 1_000_000

start = time.perf_counter()

result_chunked = numpy_chunked(
    N,
    chunk_size=CHUNK_SIZE
)

chunked_time = time.perf_counter() - start

print("Result:", int(result_chunked))
print(f"Chunked NumPy: {chunked_time:.4f} s")
print(f"Speedup vs Python: {python_time / chunked_time:.2f}x")

Result: 3599999947
Chunked NumPy: 0.9373 s
Speedup vs Python: 11.39x


## 6. Check correctness

All implementations should return exactly the same result.

In [17]:
print("Pure Python :", result_python)
print("NumPy       :", int(result_numpy))
print("Chunked     :", int(result_chunked))

print()
print(
    "All equal:",
    result_python == int(result_numpy) == int(result_chunked)
)

Pure Python : 3599999947
NumPy       : 3599999947
Chunked     : 3599999947

All equal: True


## 7. Compare execution times

In [18]:
print(f"Pure Python   : {python_time:.4f} s")
print(f"NumPy         : {numpy_time:.4f} s")
print(f"Chunked NumPy : {chunked_time:.4f} s")

print()
print(f"NumPy speedup         : {python_time / numpy_time:.2f}x")
print(f"Chunked NumPy speedup : {python_time / chunked_time:.2f}x")

Pure Python   : 10.6773 s
NumPy         : 2.2975 s
Chunked NumPy : 0.9373 s

NumPy speedup         : 4.65x
Chunked NumPy speedup : 11.39x


## 8. Why chunking is useful even if it is not faster

Chunking is not necessarily faster than processing the entire array at once.

It may even be slightly slower because Python must manage several chunks.

Its main advantages are:

- **lower peak memory usage**;
- ability to process datasets larger than available RAM;
- natural decomposition into independent units of work;
- easier transition to **parallel execution**.

So chunking is often a scalability technique rather than a pure speed optimization.

## 9. Chunks are naturally parallelizable

Our chunks are independent.

For example:

```text
Chunk 1 → process values       0 ...  999,999
Chunk 2 → process values 1,000,000 ... 1,999,999
Chunk 3 → process values 2,000,000 ... 2,999,999
...
```

Each chunk produces only a **partial sum**.

At the end:

```text
partial_1
partial_2
partial_3
   ...
    ↓
sum everything
    ↓
final result
```

Because one chunk does not depend on the result of another chunk, they can potentially be assigned to different CPU cores or worker processes.

### Sequential chunks

Our current implementation does this:

```text
CPU

chunk 1: ███████
chunk 2:        ███████
chunk 3:               ███████
chunk 4:                      ███████
```

The chunks are independent, but we still process them one after another.

### Parallel chunks

Conceptually, with multiple worker processes:

```text
Core 1: chunk 1 ███████    chunk 3 ███████
Core 2: chunk 2 ███████    chunk 4 ███████
```

Each worker can compute a partial result independently.

This is a good situation for multiprocessing because:

1. the work is naturally divisible;
2. there is very little communication between workers;
3. only the final partial sums need to be combined.

## 10. A chunk worker function

We can explicitly represent the operation performed on one chunk: see `efficient_chunking.py`

# Take-home messages

- Pure Python loops are flexible, but millions of Python-level iterations can be expensive.
- NumPy is fast because numerical operations are implemented in optimized compiled code.
- Full vectorization is often the simplest and fastest solution when the arrays comfortably fit in memory.
- Chunking reduces peak memory usage.
- Chunking can process datasets that are too large to fit into memory at once.
- Independent chunks are also **natural units of parallel work**.
- Parallelism introduces overhead, so the computation inside each chunk should be sufficiently substantial.
- A useful progression is:

```text
Pure Python
    ↓
NumPy vectorization
    ↓
Chunked NumPy when memory becomes an issue
    ↓
Parallelize independent chunks when useful
```

The key optimization principle is:

> **First move expensive work out of the Python interpreter. Then think about parallelism.**